# Memory 

Personal Assistant with 3 memory **working / episodic / semantic"


## 1. Imports & Schema (`Memory` dataclass)

In [125]:
from __future__ import annotations

import re
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from enum import Enum
from typing import Optional


class MemoryType(str, Enum):
    WORKING = "working"
    EPISODIC = "episodic"
    SEMANTIC = "semantic"


def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


@dataclass
class Memory:
    user_id: str
    memory_type: MemoryType
    content: str
    created_at: str = field(default_factory=_now_iso)
    confidence: float = 1.0
    memory_id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    # working-memory-only fields
    session_id: Optional[str] = None
    ttl_expires_at: Optional[str] = None
    status: Optional[str] = None

    # episodic-only fields
    event_type: Optional[str] = None
    related_task_id: Optional[str] = None

    # semantic-only fields
    category: Optional[str] = None
    evidence_count: int = 1
    last_reinforced_at: Optional[str] = None
    polarity: Optional[str] = None          # "prefer" | "reject"
    is_hard_constraint: bool = False

    def to_dict(self) -> dict:
        d = asdict(self)
        d["memory_type"] = self.memory_type.value
        return d

    def __str__(self) -> str:
        base = f"[{self.memory_type.value:>8}] ({self.confidence:.2f}) {self.content}"
        if self.memory_type == MemoryType.SEMANTIC and self.is_hard_constraint:
            base += "  <HARD CONSTRAINT>"
        return base

## 2. should_store_memory(message) -> bool

Nguyên tắc: mặc định KHÔNG lưu. Chỉ lưu khi message mang thông tin có giá trị lâu dài (declarative preference/constraint, correction, fact định danh, hoặc kết quả một hành động đã hoàn tất). Câu hỏi thuần, small talk, task one-off, và câu chưa chắc chắn đều không được persit vào long term memory.

In [126]:
QUESTION_PATTERNS = [
    r"\?\s*$",
    r"^(giá|bao nhiêu|khi nào|ở đâu|ai|làm sao|thế nào)\b",
]
SMALLTALK_PATTERNS = [
    r"cảm ơn", r"cám ơn", r"chào bạn", r"bạn giỏi", r"hihi", r"ok bạn",
]
HEDGE_PATTERNS = [
    r"có lẽ", r"chưa chắc", r"để xem", r"hình như", r"có thể sẽ",
]
CORRECTION_PATTERNS = [
    r"lần trước.*(sai|nhầm|lỗi)",
    r"nhớ.*(check|kiểm tra)",
    r"đừng.*(nữa)\b",
]
DECLARATIVE_PREF_PATTERNS = [
    r"\btôi thích\b", r"\btôi không thích\b", r"\btôi luôn\b",
    r"\bkhông bao giờ\b", r"\bưu tiên\b", r"\btôi là người\b",
    r"\bđừng gợi ý\b", r"\bkhông chấp nhận\b",
]
ONE_OFF_PATTERNS = [
    r"\bhôm nay\b", r"\btối nay\b", r"\blần này thôi\b", r"\bchỉ lần này\b",
]
COMPLETED_ACTION_PATTERNS = [
    r"\bđã đặt\b", r"\bđã hoàn tất\b", r"\bđã xác nhận\b",
]


def _matches_any(patterns: list[str], text: str) -> bool:
    text_l = text.lower()
    return any(re.search(p, text_l) for p in patterns)


def should_store_memory(message: str, context: Optional[dict] = None) -> bool:
    """Quyet dinh xem message co dang luu vao long-term memory (episodic/
    semantic) hay khong. Working memory (task hien tai) luon duoc giu tam
    thoi trong session bat ke ham nay tra ve gi - ham nay chi gac cong cho
    LONG-TERM persistence."""
    context = context or {}
    text = message.strip()

    # 1. Cau hoi thuan tuy, khong chua thong tin ve user -> bo
    if _matches_any(QUESTION_PATTERNS, text) and not _matches_any(DECLARATIVE_PREF_PATTERNS, text):
        return False

    # 2. Smalltalk / filler -> bo
    if _matches_any(SMALLTALK_PATTERNS, text):
        return False

    # 3. Correction - uu tien cao, luu ngay
    if _matches_any(CORRECTION_PATTERNS, text):
        return True

    # 4. Hedge / chua chac chan -> chua luu
    if _matches_any(HEDGE_PATTERNS, text):
        return False

    # 5. Phat bieu preference/constraint declarative -> luu
    if _matches_any(DECLARATIVE_PREF_PATTERNS, text):
        return True

    # 6. Ket qua mot hanh dong da hoan tat -> luu (episodic)
    if _matches_any(COMPLETED_ACTION_PATTERNS, text):
        return True

    # 7. Task one-off, gioi han thoi gian -> bo
    if _matches_any(ONE_OFF_PATTERNS, text):
        return False

    # Mac dinh: khong luu
    return False

## 3. classifiy_memory - phân loại episodic vs semantic + gán field phù hợp

In [127]:
def classify_memory(message: str, user_id: str, session_id: str) -> Optional[Memory]:
    """Neu should_store_memory == True, phan loai tiep xem la episodic hay
    semantic, va gan cac field phu hop (polarity, hard_constraint, category)."""
    if not should_store_memory(message):
        return None

    text = message.lower()

    if _matches_any(COMPLETED_ACTION_PATTERNS, text):
        return Memory(
            user_id=user_id,
            memory_type=MemoryType.EPISODIC,
            content=message.strip(),
            event_type="action_completed",
        )

    if _matches_any(CORRECTION_PATTERNS, text):
        return Memory(
            user_id=user_id,
            memory_type=MemoryType.SEMANTIC,
            content=message.strip(),
            category="operating_instruction",
            confidence=0.9,
            polarity="prefer",
        )

    # declarative preference/constraint
    is_negative = bool(re.search(r"không thích|không bao giờ|không chấp nhận|đừng", text))
    is_hard = bool(re.search(r"không chấp nhận|không bao giờ|quá \d+", text))
    return Memory(
        user_id=user_id,
        memory_type=MemoryType.SEMANTIC,
        content=message.strip(),
        category="flight_preference" if "bay" in text or "transit" in text or "chuyến" in text else "general_preference",
        confidence=0.85 if is_hard else 0.6,
        polarity="reject" if is_negative else "prefer",
        is_hard_constraint=is_hard,
    )

## 4. MemoryStore - lưu trữ in-memory, có reinforce cho semantic memory

In [128]:
class MemoryStore:
    def __init__(self):
        self._memories: list[Memory] = []

    def add(self, memory: Memory) -> Memory:
        # Neu la semantic va da co memory tuong tu (cung category+polarity),
        # thi reinforce (tang confidence + evidence_count) thay vi tao ban ghi moi.
        if memory.memory_type == MemoryType.SEMANTIC:
            existing = self._find_similar_semantic(memory)
            if existing:
                existing.evidence_count += 1
                existing.confidence = min(1.0, existing.confidence + 0.15)
                existing.last_reinforced_at = _now_iso()
                return existing
        self._memories.append(memory)
        return memory

    def _find_similar_semantic(self, memory: Memory) -> Optional[Memory]:
        for m in self._memories:
            if (
                m.memory_type == MemoryType.SEMANTIC
                and m.category == memory.category
                and m.polarity == memory.polarity
            ):
                return m
        return None

    def add_working(self, user_id: str, session_id: str, content: str, ttl_minutes: int = 60) -> Memory:
        m = Memory(
            user_id=user_id,
            memory_type=MemoryType.WORKING,
            content=content,
            session_id=session_id,
            ttl_expires_at=(datetime.now(timezone.utc) + timedelta(minutes=ttl_minutes)).isoformat(),
            status="in_progress",
        )
        self._memories.append(m)
        return m

    def query(self, user_id: str, memory_type: Optional[MemoryType] = None) -> list[Memory]:
        return [
            m for m in self._memories
            if m.user_id == user_id and (memory_type is None or m.memory_type == memory_type)
        ]

    def dump(self, user_id: str) -> None:
        print(f"\n--- Memory store cho user '{user_id}' ---")
        for t in (MemoryType.WORKING, MemoryType.EPISODIC, MemoryType.SEMANTIC):
            items = self.query(user_id, t)
            if not items:
                continue
            print(f"\n{t.value.upper()}:")
            for m in items:
                print(f"  {m}")

## 5. PersonalAssitantAgent - demo xử lý  hội thoại

Xử lý message -> luôn ghi working memory tạm thời -> hỏi should_store_memory -> nếu True thì classify_memory rồi lưu vào store. Đồng thời demo dùng semantic memory để lọc cứng + rank chuyến bay

In [129]:
class PersonalAssistantAgent:
    def __init__(self, user_id: str, session_id: str):
        self.user_id = user_id
        self.session_id = session_id
        self.store = MemoryStore()

    def process_message(self, message: str) -> None:
        print(f"\nUser: {message}")

        # Working memory: moi message lien quan task hien tai deu duoc giu tam
        self.store.add_working(self.user_id, self.session_id, message)

        decision = should_store_memory(message)
        print(f"  -> should_store_memory = {decision}")

        if decision:
            mem = classify_memory(message, self.user_id, self.session_id)
            stored = self.store.add(mem)
            print(f"  -> luu vao long-term: {stored}")

    def search_flights(self, candidates: list[dict]) -> list[dict]:
        """Dung semantic memory (constraint + preference) de loc va rank cac
        chuyen bay ung vien."""
        semantic = self.store.query(self.user_id, MemoryType.SEMANTIC)

        max_transit = None
        prefers_morning = False
        for m in semantic:
            if m.category == "flight_preference" and m.is_hard_constraint and "transit" in m.content.lower():
                match = re.search(r"(\d+)\s*tiếng", m.content)
                if match:
                    max_transit = int(match.group(1))
            if m.category == "flight_preference" and m.polarity == "prefer" and "sáng" in m.content.lower():
                prefers_morning = True

        # 1. Filter cung theo hard constraint
        filtered = [
            c for c in candidates
            if max_transit is None or c["transit_hours"] <= max_transit
        ]

        # 2. Rank theo soft preference (bay sang len truoc)
        if prefers_morning:
            filtered.sort(key=lambda c: (c["departure_hour"] > 11, c["transit_hours"]))
        else:
            filtered.sort(key=lambda c: c["transit_hours"])

        return filtered

## 6. Test harness - 10 test case cho should_store_memory

In [130]:
TEST_CASES = [
    ("Tôi thích chuyến bay buổi sáng.", True),
    ("Tuần sau đặt giúp tôi chuyến đi Bangkok.", False),  # chi la task/working, khong phai preference lau dai
    ("À tôi không thích transit quá 2 tiếng.", True),
    ("Vé máy bay hôm nay giá bao nhiêu?", False),
    ("Cảm ơn nha, bạn giỏi quá.", False),
    ("Chuyến trước bạn đặt nhầm ngày, lần này nhớ check kỹ nha.", True),
    ("Có lẽ tôi sẽ đổi sang ăn chay, chưa chắc.", False),
    ("Tôi là người ăn chay, đừng gợi ý đồ ăn có thịt.", True),
    ("Đặt cho tôi bàn ăn tối nay lúc 7h thôi.", False),
    ("Tôi luôn bay Vietnam Airlines vì tích điểm thành viên.", True),
]


def run_tests() -> None:
    print("=== TEST should_store_memory ===")
    passed = 0
    for i, (msg, expected) in enumerate(TEST_CASES, 1):
        actual = should_store_memory(msg)
        ok = actual == expected
        passed += ok
        status = "PASS" if ok else "FAIL"
        print(f"{i:>2}. [{status}] expected={expected!s:<5} actual={actual!s:<5} | {msg}")
    print(f"\n{passed}/{len(TEST_CASES)} test case pass.")


run_tests()

=== TEST should_store_memory ===
 1. [PASS] expected=True  actual=True  | Tôi thích chuyến bay buổi sáng.
 2. [PASS] expected=False actual=False | Tuần sau đặt giúp tôi chuyến đi Bangkok.
 3. [PASS] expected=True  actual=True  | À tôi không thích transit quá 2 tiếng.
 4. [PASS] expected=False actual=False | Vé máy bay hôm nay giá bao nhiêu?
 5. [PASS] expected=False actual=False | Cảm ơn nha, bạn giỏi quá.
 6. [PASS] expected=True  actual=True  | Chuyến trước bạn đặt nhầm ngày, lần này nhớ check kỹ nha.
 7. [PASS] expected=False actual=False | Có lẽ tôi sẽ đổi sang ăn chay, chưa chắc.
 8. [PASS] expected=True  actual=True  | Tôi là người ăn chay, đừng gợi ý đồ ăn có thịt.
 9. [PASS] expected=False actual=False | Đặt cho tôi bàn ăn tối nay lúc 7h thôi.
10. [PASS] expected=True  actual=True  | Tôi luôn bay Vietnam Airlines vì tích điểm thành viên.

10/10 test case pass.


## 7. Demo end-to-end: xử lý hội thoại

In [131]:
agent = PersonalAssistantAgent(user_id="u_123", session_id="s_456")

conversation = [
    "Tôi thích chuyến bay buổi sáng.",
    "Tuần sau đặt giúp tôi chuyến đi Bangkok.",
    "À tôi không thích transit quá 2 tiếng.",
]
for msg in conversation:
    agent.process_message(msg)

agent.store.dump("u_123")


User: Tôi thích chuyến bay buổi sáng.
  -> should_store_memory = True
  -> luu vao long-term: [semantic] (0.60) Tôi thích chuyến bay buổi sáng.

User: Tuần sau đặt giúp tôi chuyến đi Bangkok.
  -> should_store_memory = False

User: À tôi không thích transit quá 2 tiếng.
  -> should_store_memory = True
  -> luu vao long-term: [semantic] (0.85) À tôi không thích transit quá 2 tiếng.  <HARD CONSTRAINT>

--- Memory store cho user 'u_123' ---

WORKING:
  [ working] (1.00) Tôi thích chuyến bay buổi sáng.
  [ working] (1.00) Tuần sau đặt giúp tôi chuyến đi Bangkok.
  [ working] (1.00) À tôi không thích transit quá 2 tiếng.

SEMANTIC:
  [semantic] (0.60) Tôi thích chuyến bay buổi sáng.
  [semantic] (0.85) À tôi không thích transit quá 2 tiếng.  <HARD CONSTRAINT>


# 8. Demo: Dùng semantic memory để lọc/rank chuyến bay


In [132]:
candidate_flights = [
    {"flight": "VN601", "departure_hour": 7, "transit_hours": 1.5},
    {"flight": "TG320", "departure_hour": 14, "transit_hours": 1.0},
    {"flight": "AK890", "departure_hour": 6, "transit_hours": 3.0},   # bi loai vi transit > 2h
    {"flight": "PG112", "departure_hour": 9, "transit_hours": 2.0},
]

ranked = agent.search_flights(candidate_flights)
for f in ranked:
    print(f"{f['flight']}: khoi hanh {f['departure_hour']}h, transit {f['transit_hours']}h")

VN601: khoi hanh 7h, transit 1.5h
PG112: khoi hanh 9h, transit 2.0h
TG320: khoi hanh 14h, transit 1.0h


# 9. Ghi chú/ hạn chế cần lưu ý (cho bài 8-9, failure taxnonomy)
- Regex tiếng Việt dễ vợ với cách diễn đạt lệch mẫu -> productin nên dùng LLM classifer nhỏ cho case mơ hồ, giữ regex làm fast path
- _find_similar_semantic so khớp bằng category + polarity - quá thô, dễ gộp nhầm 2 preference khác nội dung; cần semantic similarity (embedding) thực sự
- working memory chưa có job dọn dẹp khi ttl_expires_at hết hạn - mới lưu timestmap, chưa xóa tự động.